In [0]:
source_path_day1 = "/Volumes/scd_inc_catalog/scd_schema/scd_vol/day1/customers_day1.csv"

In [0]:
source_day1 = spark.read.csv(source_path_day1, header=True,inferSchema=True)
source_day1.show(5)
source_day1.printSchema()
print("no of records in day 1:",source_day1.count())

In [0]:
from pyspark.sql.functions import col,regexp_replace,trim,concat,lit,initcap,upper,lower
source_day1_clean = source_day1.withColumnRenamed("email","email_id")\
    .withColumn("email_id",regexp_replace("email_id","@","_"))\
    .withColumn("first_name",trim(col("first_name")))\
    .withColumn("last_name",trim(col("last_name")))\
        .withColumn("category",trim(initcap(col("category"))))\
            .withColumn("city",trim(col("city")))\
                .withColumn("payment_method",trim(col("payment_method")))\
                .withColumn("customer_name",concat(col("first_name"),lit(" "),col("last_name")))\
                    .withColumn("email_id",regexp_replace("email_id","_","@"))\
                        .withColumn("email_id",lower(col("email_id")))\
                    .drop("first_name","last_name")

source_day1_clean = source_day1_clean.select("order_id","customer_name","email_id","city","product_id","category","amount","quantity","payment_method")
source_day1_clean.display()

In [0]:
# filtering  based on category = 'electronics'
electronics_df = source_day1_clean.filter(col("category") == "Electronics").filter(col("amount")>15000)
electronics_df.show()
electronics_df.printSchema()
electronics_df.count()

In [0]:
duplicate_orders = (
    source_day1_clean
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_orders) #displaying duplicate records in order_id

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,current_date,to_date,lit
window_spec = Window.orderBy("order_id")

#creating 4 new columns 
initial_silver = (
    source_day1_clean

    .withColumn(
        "order_sk",
        row_number().over(window_spec)
    )

    .withColumn(
        "start_date",
        current_date()
    )

    .withColumn(
        "end_date",
        to_date(lit("9999-12-31"))
    )

    .withColumn(
        "is_current",
        lit('Y')
    )
)

initial_silver.show(5)

In [0]:
initial_silver_df = initial_silver.select(
    "order_sk",
    "order_id",
    "customer_name",
    "product_id",
    "category",
    "quantity",
    "amount",
    "payment_method",
    "city",
    "email_id",
    "start_date",
    "end_date",
    "is_current"
)
initial_silver_df.show(10)
print("no of records :",initial_silver_df.count())

In [0]:
silver_path = "/Volumes/scd_inc_catalog/scd_schema/scd_vol_silver/" #here i already have created a volume inside it the delta file created by databricks

(
    initial_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)


In [0]:
silver_df = (
    spark.read
.format("delta")
.load(silver_path)
)

display(silver_df)
print("no of records in delta file:",silver_df.count())

##NOW COMES THE INCREMENTAL LOAD FOR DAY2 FILE

In [0]:
# Reading Incremental day 2 file
source_path_day2 = "/Volumes/scd_inc_catalog/scd_schema/scd_vol/day2/customers_day2.csv"
incremental_df = spark.read.csv(source_path_day2, header=True,inferSchema=True)
incremental_df.show(5)
incremental_df.printSchema()
print("no of records in day 2:",incremental_df.count())

In [0]:
incremental_df = incremental_df.withColumnRenamed("email","email_id")\
    .withColumn("first_name",trim(col("first_name")))\
    .withColumn("last_name",trim(col("last_name")))\
        .withColumn("category",trim(initcap(col("category"))))\
            .withColumn("city",trim(col("city")))\
                .withColumn("payment_method",trim(col("payment_method")))\
                    .withColumn("product_id",trim(initcap(col("product_id").cast("string"))))\
                        .withColumn("amount",col("amount").cast("double"))\
                            .withColumn("quantity",col("quantity").cast("integer"))\
                    .withColumn("customer_name",concat(col("first_name"),lit(" "),col("last_name")))\
                        .drop("first_name","last_name")\
                            .dropDuplicates(["order_id"])
                        

incremental_clean = incremental_df.select("order_id","customer_name","email_id","city","product_id","category","amount","quantity","payment_method")
incremental_clean.show(5)
#incremental_clean.printSchema()
print("no of records in incremental file:",incremental_clean.count())

In [0]:
silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

current_silver = (
    silver_df
    .filter(col("is_current") == 'Y')
)

print(
    "Current Silver records:",
    current_silver.count()
)

In [0]:
new_records = (
    incremental_clean.alias("src")
    .join(
        current_silver.alias("tgt"),
        col("src.order_id") == col("tgt.order_id"), #here LEFT JOIN is used because to find only new records(unmatched records from right table)
        "left"
    )
    .filter(
         col("tgt.order_id").isNull() #here we are filtering only new records from right table 
    )
    .select("src.*")
)

print("New records:", new_records.count())
new_records.show(5)

In [0]:
from pyspark.sql.functions import coalesce,lit
changed_records = (
    incremental_clean.alias("src")
    .join(
        current_silver.alias("tgt"),
        col("src.order_id") == col("tgt.order_id"),
        "inner"
    )
    .filter(
         (coalesce(col("src.customer_name"), lit("")) !=
         coalesce(col("tgt.customer_name"), lit("")))
         |
        (coalesce(col("src.product_id"), lit("")) !=
         coalesce(col("tgt.product_id"), lit("")))
        |
        (coalesce(col("src.category"), lit("")) !=
         coalesce(col("tgt.category"), lit("")))
        |
        (coalesce(col("src.quantity"), lit(-1)) !=
         coalesce(col("tgt.quantity"), lit(-1)))
        |
        (coalesce(col("src.amount"), lit(-1.0)) !=
         coalesce(col("tgt.amount"), lit(-1.0)))
        |
        (coalesce(col("src.payment_method"), lit("")) !=
        coalesce(col("tgt.payment_method"), lit("")))
        |
        (coalesce(col("src.city"), lit("")) !=
         coalesce(col("tgt.city"), lit("")))
        |
        (coalesce(col("src.email_id"), lit("")) !=
         coalesce(col("tgt.email_id"), lit("")))
).select("src.*")
)
print("Changed records:", changed_records.count())
changed_records.show(5)  #here This snippet performs Change Data Capture (CDC) or change detection between an incoming batch of data (incremental_clean) and an existing target table (current_silver).

#The != (not equal) operator is used because the query's objective is to find updates—rows where the incoming data has changed compared to what is currently stored in the table.

In [0]:
changed_ids = (
    changed_records
    .select("order_id","customer_name")
    .distinct()
)

display(changed_ids)

In [0]:
from pyspark.sql.functions import date_add,current_date,when,lit
 

expired_silver = (
    silver_df.alias("tgt")
    .join(
        changed_ids.alias("chg"),
        col("tgt.order_id") == col("chg.order_id"),
        "left"
    )
    .withColumn(
        "end_date",
        when(
            col("chg.order_id").isNotNull() &
            (col("tgt.is_current") == 'Y'),
            date_add (current_date(), 1)
        )
        .otherwise(col("tgt.end_date"))
    )
    .withColumn(
        "is_current",
        when(
            col("chg.order_id").isNotNull() &
            (col("tgt.is_current") == 'Y'),
            lit('N')
        )
        .otherwise(col("tgt.is_current"))
    )
    .select(
        "tgt.order_sk",
        "tgt.order_id",
        "tgt.customer_name",
        "tgt.product_id",
        "tgt.category",
        "tgt.quantity",
        "tgt.amount",
        "tgt.payment_method",
        "tgt.city",
        "tgt.email_id",
        "tgt.start_date",
        "end_date",
        "is_current"
    )
)
display(expired_silver)

In [0]:
records_to_insert = (
    new_records
    .unionByName(changed_records)
)
print(
    "Records to insert:",
    records_to_insert.count()
) 

In [0]:
from pyspark.sql.functions import aggregate,max,min,avg
max_sk = (
    silver_df
    .agg(max("order_sk").alias("max_sk"))
    .first()["max_sk"]
)

print("Current maximum surrogate key:", max_sk)

In [0]:
new_records_window = Window.orderBy("order_id")

new_versions = (
    records_to_insert

    .withColumn(
        "order_sk",
        row_number().over(new_records_window) + max_sk
    )

    .withColumn(
        "start_date",
        current_date()
    )

    .withColumn(
        "end_date",
        to_date(lit("9999-12-31"))
    )

    .withColumn(
        "is_current",
        lit('Y')
    )

    .select(
        "order_sk",
        "order_id",
        "customer_name",
        "product_id",
        "category",
        "quantity",
        "amount",
        "payment_method",
        "city",
        "email_id",
        "start_date",
        "end_date",
        "is_current"
    )
)

print("no of new records:",new_versions.count())

In [0]:
final_silver_df = (
    expired_silver
    .unionByName(new_versions)
)

print(
    "Final Silver record count:",
    final_silver_df.count()
)

In [0]:
print(
    "Current records:",
    final_silver_df
    .filter(col("is_current") == 'Y')
    .count()
)

In [0]:
(
    final_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

In [0]:
final_silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)
final_silver_df.display()
print("no of records :",final_silver_df.count()) # total records 1400 i.e, 1300 day 2 file and 100 incremental after joining day 1 silver file to day 2 incremental file

In [0]:
final_silver_df = spark.read.format("delta").load(silver_path)
final_silver_df = final_silver_df.filter(col("is_current") == 'N')
final_silver_df.display()

In [0]:
source_path_day3 = "/Volumes/scd_inc_catalog/scd_schema/scd_vol/day3/customers_day3.csv"

In [0]:
incremental_day3_df = spark.read.csv(source_path_day3, header=True,inferSchema=True)
incremental_day3_df.show(5)
incremental_day3_df.printSchema()
print("no of records in day 3:",incremental_day3_df.count())

In [0]:
incremental_day3_df = incremental_day3_df.withColumnRenamed("email","email_id")\
    .withColumn("first_name",trim(col("first_name")))\
    .withColumn("last_name",trim(col("last_name")))\
        .withColumn("category",trim(initcap(col("category"))))\
            .withColumn("city",trim(col("city")))\
                .withColumn("payment_method",trim(col("payment_method")))\
                    .withColumn("product_id",trim(initcap(col("product_id").cast("string"))))\
                        .withColumn("amount",col("amount").cast("double"))\
                            .withColumn("quantity",col("quantity").cast("integer"))\
                    .withColumn("customer_name",concat(col("first_name"),lit(" "),col("last_name")))\
                        .drop("first_name","last_name")\
                            .dropDuplicates(["order_id"])
                        

incremental_day3_df = incremental_day3_df.select("order_id","customer_name","email_id","city","product_id","category","amount","quantity","payment_method")
incremental_day3_df.show(5)
#incremental_clean.printSchema()
print("no of records in incremental file:",incremental_day3_df.count())

In [0]:
final_silver_df = (
    spark.read
    .format("delta")
    .load(silver_path_day2)
)
current_silver_day3 = (
    final_silver_df.filter(col("is_current") == 'Y')
)

print("no of records :",current_silver_day3.count()) 
current_silver_day3.display()

In [0]:
new_records_day3 = (
    incremental_day3_df.alias("src")
    .join(
        current_silver_day3.alias("tgt"),
        col("src.order_id") == col("tgt.order_id"), #here LEFT JOIN is used because to find only new records(unmatched records from right table)
        "left"
    )
    .filter(
         col("tgt.order_id").isNull() #here we are filtering only new records from right table 
    )
    .select("src.*")
)

print("New records:", new_records_day3.count())
new_records_day3.show(5)
     

In [0]:

from pyspark.sql.functions import coalesce,lit
changed_records_day3 = (
    incremental_day3_df.alias("src")
    .join(
        current_silver_day3.alias("tgt"),
        col("src.order_id") == col("tgt.order_id"),
        "inner"
    )
    .filter(
         (coalesce(col("src.customer_name"), lit("")) !=
         coalesce(col("tgt.customer_name"), lit("")))
         |
        (coalesce(col("src.product_id"), lit("")) !=
         coalesce(col("tgt.product_id"), lit("")))
        |
        (coalesce(col("src.category"), lit("")) !=
         coalesce(col("tgt.category"), lit("")))
        |
        (coalesce(col("src.quantity"), lit(-1)) !=
         coalesce(col("tgt.quantity"), lit(-1)))
        |
        (coalesce(col("src.amount"), lit(-1.0)) !=
         coalesce(col("tgt.amount"), lit(-1.0)))
        |
        (coalesce(col("src.payment_method"), lit("")) !=
        coalesce(col("tgt.payment_method"), lit("")))
        |
        (coalesce(col("src.city"), lit("")) !=
         coalesce(col("tgt.city"), lit("")))
        |
        (coalesce(col("src.email_id"), lit("")) !=
         coalesce(col("tgt.email_id"), lit("")))
).select("src.*")
)
print("Changed records for day 3:", changed_records_day3.count())
changed_records_day3.show(5)

In [0]:

changed_ids_day3 = (
    changed_records_day3
    .select("order_id")
    .distinct()
)

display(changed_ids_day3)

In [0]:

from pyspark.sql.functions import date_add,current_date,when,lit
 

expired_silver_day3 = (
    final_silver_df.alias("tgt")
    .join(
        changed_ids_day3.alias("chg"),
        col("tgt.order_id") == col("chg.order_id"),
        "left"
    )
    .withColumn(
        "end_date",
        when(
            col("chg.order_id").isNotNull() &
            (col("tgt.is_current") == 'Y'),
            date_add (current_date(), 1)
        )
        .otherwise(col("tgt.end_date"))
    )
    .withColumn(
        "is_current",
        when(
            col("chg.order_id").isNotNull() &
            (col("tgt.is_current") == 'Y'),
            lit('N')
        )
        .otherwise(col("tgt.is_current"))
    )
    .select(
        "tgt.order_sk",
        "tgt.order_id",
        "tgt.customer_name",
        "tgt.product_id",
        "tgt.category",
        "tgt.quantity",
        "tgt.amount",
        "tgt.payment_method",
        "tgt.city",
        "tgt.email_id",
        "tgt.start_date",
        "end_date",
        "is_current"
    )
)
display(expired_silver_day3)

In [0]:

records_to_insert_day3 = (
    new_records_day3
    .unionByName(changed_records_day3)
)
print(
    "Records to insert:",
    records_to_insert_day3.count()
) 

In [0]:
from pyspark.sql.functions import aggregate,max,min,avg
max_sk_3 = (
    final_silver_df
    .agg(max("order_sk").alias("max_sk"))
    .first()["max_sk"]
)

print("Current maximum surrogate key:", max_sk_3)

In [0]:
new_records_window_day3 = Window.orderBy("order_id")

new_versions_day3 = (
    records_to_insert_day3

    .withColumn(
        "order_sk",
        row_number().over(new_records_window_day3) + max_sk_3
    )

    .withColumn(
        "start_date",
        current_date()
    )

    .withColumn(
        "end_date",
        to_date(lit("9999-12-31"))
    )

    .withColumn(
        "is_current",
        lit('Y')
    )

    .select(
        "order_sk",
        "order_id",
        "customer_name",
        "product_id",
        "category",
        "quantity",
        "amount",
        "payment_method",
        "city",
        "email_id",
        "start_date",
        "end_date",
        "is_current"
    )
)

print("no of new records:",new_versions_day3.count())


In [0]:
final_silver_df_day3 = (
    expired_silver_day3
    .unionByName(new_versions_day3)
)

print(
    "Final Silver record count:",
    final_silver_df_day3.count()
)

In [0]:
print(
    "Current records:",
    final_silver_df_day3
    .filter(col("is_current") == 'Y')
    .count()
)
     

In [0]:

(
    final_silver_df_day3
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)
final_silver_df_day3.count()

In [0]:

final_silver_df_day3 = (
    spark.read
    .format("delta")
    .load(silver_path)
)
final_silver_df_day3.display()
print("no of records :",final_silver_df_day3.count())

In [0]:
final_silver_df_day3 = final_silver_df_day3.filter(col("is_current") == 'N')
final_silver_df_day3.display()
print("no of records :",final_silver_df_day3.count())